In [3]:
import pandas as pd

df = pd.read_csv('../data/raw/metric_snapshots.csv')
display(df)

,schemaVersion,runId,entityId,entityType,observedAt,status,utilizationPct,latencyMs,packetLossPct,throughputMbps,source,sequence
0,1.0,demo-normal-to-congested,s1-s2,link,2026-01-01T00:00:00Z,healthy,62.0,20.0,0.0,6.2,fixture,0
1,1.0,demo-normal-to-congested,s1-s2,link,2026-01-01T00:00:05Z,degraded,82.0,55.0,2.0,8.2,fixture,1
2,1.0,demo-normal-to-congested,s1-s2,link,2026-01-01T00:00:10Z,degraded,84.0,57.0,2.5,8.4,fixture,2
3,1.0,demo-normal-to-congested,s1-s2,link,2026-01-01T00:00:15Z,degraded,83.0,56.0,2.0,8.3,fixture,3
4,1.0,demo-normal-to-congested,s1-s2,link,2026-01-01T00:00:20Z,healthy,60.0,25.0,0.0,6.0,fixture,4


In [4]:
import pandas as pd

# تحويل عمود الوقت إلى صيغة قابلة للحساب
df['observedAt'] = pd.to_datetime(df['observedAt'], utc=True, errors='coerce')
decision_time = pd.Timestamp("2026-01-01T00:00:25Z")

# 1. حساب عمر البيانات بالثواني
df['telemetryAgeSeconds'] = (decision_time - df['observedAt']).dt.total_seconds()

# 2. حساب التغير في الاستهلاك مقارنة بالقراءة السابقة
df['deltaUtilizationPct'] = df.groupby(['runId', 'entityId'])['utilizationPct'].diff()

# 3. تحديد وجود ازدحام بناءً على الشروط المتفق عليها
df['congestionSignal'] = (
    ((df['utilizationPct'].notna()) & (df['utilizationPct'] >= 80.0)) |
    ((df['latencyMs'].notna()) & (df['latencyMs'] >= 50.0)) |
    ((df['packetLossPct'].notna()) & (df['packetLossPct'] >= 2.0))
)

display(df[['sequence', 'telemetryAgeSeconds', 'deltaUtilizationPct', 'congestionSignal']])

,sequence,telemetryAgeSeconds,deltaUtilizationPct,congestionSignal
0,0,25.0,NaN,False
1,1,20.0,20.0,True
2,2,15.0,2.0,True
3,3,10.0,-1.0,True
4,4,5.0,-23.0,False


In [5]:
from dataclasses import dataclass
from typing import Any

# إعدادات محرك القرارات (الحدود المسموحة للازدحام والتأخير)
@dataclass(frozen=True)
class RuleConfig:
    staleAfterSeconds: float = 12.0
    highUtilizationPct: float = 80.0
    clearUtilizationPct: float = 70.0
    highLatencyMs: float = 50.0
    clearLatencyMs: float = 35.0
    highPacketLossPct: float = 2.0
    clearPacketLossPct: float = 1.0
    requiredWindows: int = 3
    cooldownSeconds: float = 15.0
    policyVersion: str = "rules-1.0"

def decide(snapshot: dict, state: dict, now_s: float, cfg: RuleConfig = RuleConfig()) -> tuple:
    entity_id = snapshot["entityId"]
    state = {"highCount": 0, "clearCount": 0, "engineStatus": "normal", **state}
    reasons = []
    
    age = snapshot.get("telemetryAgeSeconds")
    values = [snapshot.get("utilizationPct"), snapshot.get("latencyMs"), snapshot.get("packetLossPct")]
    
    stale = age is None or age > cfg.staleAfterSeconds
    failed = snapshot.get("status") == "failed"
    
    if failed:
        state["engineStatus"] = "failureSuspected"
        solution_id = "RECOVERY_REQUEST"
        reasons.append("STATUS_FAILED")
    elif stale or all(v is None for v in values):
        state["engineStatus"] = "unknown"
        solution_id = "HOLD"
        reasons.append("STALE_OR_INCOMPLETE_TELEMETRY")
    else:
        high = (
            (values[0] is not None and values[0] >= cfg.highUtilizationPct) or
            (values[1] is not None and values[1] >= cfg.highLatencyMs) or
            (values[2] is not None and values[2] >= cfg.highPacketLossPct)
        )
        clear = (
            all(v is not None for v in values) and
            values[0] < cfg.clearUtilizationPct and values[1] < cfg.clearLatencyMs and values[2] < cfg.clearPacketLossPct
        )
        
        state["highCount"] = state["highCount"] + 1 if high else 0
        state["clearCount"] = state["clearCount"] + 1 if clear else 0
        
        if state["engineStatus"] == "congested" and state["clearCount"] >= cfg.requiredWindows:
            state["engineStatus"] = "normal"
            solution_id = "NOOP"
            reasons.append("CONGESTION_CLEARED")
        elif state["highCount"] >= cfg.requiredWindows:
            state["engineStatus"] = "congested"
            solution_id = "REROUTE_REQUEST"
            reasons.append("HIGH_CONGESTION_FEATURE")
        elif state["engineStatus"] == "congested":
            solution_id = "HOLD"
            reasons.append("WAIT_FOR_HYSTERESIS_CLEAR")
        elif state["highCount"] > 0:
            solution_id = "HOLD"
            reasons.append("WAIT_FOR_CONGESTION_CONFIRMATION")
        else:
            state["engineStatus"] = "normal"
            solution_id = "NOOP"
            reasons.append("NO_TRIGGER")

    decision = {
        "solutionId": solution_id,
        "engineStatus": state["engineStatus"],
        "reasonCodes": sorted(set(reasons))
    }
    return decision, state

# اختبار المحرك على البيانات التي قرأناها في الخطوة السابقة
current_state = {}
for index, row in df.iterrows():
    decision, current_state = decide(row.to_dict(), current_state, 0.0)
    print(f"Sequence {row['sequence']} -> Decision: {decision['solutionId']} | Status: {decision['engineStatus']} | Reason: {decision['reasonCodes']}")

Sequence 0 -> Decision: HOLD | Status: unknown | Reason: ['STALE_OR_INCOMPLETE_TELEMETRY']
Sequence 1 -> Decision: HOLD | Status: unknown | Reason: ['STALE_OR_INCOMPLETE_TELEMETRY']
Sequence 2 -> Decision: HOLD | Status: unknown | Reason: ['STALE_OR_INCOMPLETE_TELEMETRY']
Sequence 3 -> Decision: HOLD | Status: unknown | Reason: ['WAIT_FOR_CONGESTION_CONFIRMATION']
Sequence 4 -> Decision: NOOP | Status: normal | Reason: ['NO_TRIGGER']
